In [1]:
import requests
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import datetime

### Extract data from COINDESK API 
Test on 1 date first to see connection

In [2]:
url = 'https://data-api.coindesk.com/news/v1/article/list'

In [ ]:
# test response on a date first
try: 
    response =  requests.get(url, timeout = 10, 
                             params={
                                 "lang":"EN",
                                 "limit":1,
                                 "to_ts":1774011746,
                                 "categories":"BTC",
                                 "source_ids":"coindesk",
                                 "api_key":"xxxxxxxxx"
                             },
                             headers={"Content-type":"application/json; charset=UTF-8"}
                            ) 
    response.raise_for_status() 
    print(f'Status : {response.status_code}') 
    
except requests.exceptions.RequestException as e : 
    print(f'Error message : {e}') 

Status : 200


In [5]:
content = response.json()

In [6]:
df = pd.DataFrame(content['Data']) 

In [7]:
df.head(2)

,TYPE,ID,GUID,PUBLISHED_ON,PUBLISHED_ON_NS,IMAGE_URL,TITLE,SUBTITLE,AUTHORS,URL,...,LANG,UPVOTES,DOWNVOTES,SCORE,SENTIMENT,STATUS,CREATED_ON,UPDATED_ON,SOURCE_DATA,CATEGORY_DATA
0,121,59471809,e675fea3-208e-4a52-8224-2ca3cbaa6c27,1774006908,None,https://resources.cryptocompare.com/news/5/594...,"Bitcoin holds steady, with one analyst seeing ...","Your day-ahead look for March 20, 2026",Francisco Rodrigues,https://www.coindesk.com/daybook-us/2026/03/20...,...,EN,0,0,0,NEUTRAL,ACTIVE,1774007029,1774007030,"{'TYPE': '120', 'ID': 5, 'SOURCE_KEY': 'coinde...","[{'TYPE': '122', 'ID': 14, 'NAME': 'BTC', 'CAT..."


### Date range for data extraction

In [28]:
# compute a date range for 2024-01-01 to 2024-12-18 
extraction_dates =  pd.date_range(start = '2017-01-01', end = '2017-12-28', freq = '3D') 
df_extraction_dates =  extraction_dates.to_frame(index = False, name = 'date') 

In [22]:
df_extraction_dates['date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 120 entries, 0 to 119
Series name: date
Non-Null Count  Dtype         
--------------  -----         
120 non-null    datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 1.1 KB


In [29]:
df_extraction_dates

,date
0,2017-01-01
1,2017-01-04
2,2017-01-07
3,2017-01-10
4,2017-01-13
...,...
116,2017-12-15
117,2017-12-18
118,2017-12-21
119,2017-12-24


In [30]:
# compute unix timestamp 
df_extraction_dates['unix_timestamp'] =  df_extraction_dates['date'].apply(
    lambda d : int(datetime.datetime.combine(d, datetime.time(0,0,0)).timestamp()))

In [31]:
date_list = df_extraction_dates['unix_timestamp'].to_list()

In [32]:
print(len(date_list))

121


In [33]:
rows = []
for date in date_list : 
    response =  requests.get(url, timeout = 20, 
                             params={
                                 "lang":"EN",
                                 "limit":60,
                                 "to_ts":date,
                                 "categories":"BTC",
                                 "source_ids":"coindesk",
                                 "api_key":"6fe8401fa3326293b92b5c0b367e6f9ba806ee3a9fea1bf5a2a7533d746a56b5"
                             },
                             headers={"Content-type":"application/json; charset=UTF-8"}
                            )  
    content = response.json() 
    rows.extend(content['Data'])
    
    current_date = datetime.datetime.fromtimestamp(date) 
    data_row_count = len(content['Data'])
    print(f'Current execution data :{current_date} , count :{data_row_count}, status : completed')
    
df_real = pd.json_normalize(rows)

Current execution data :2017-01-01 00:00:00 , count :60, status : completed
Current execution data :2017-01-04 00:00:00 , count :60, status : completed
Current execution data :2017-01-07 00:00:00 , count :60, status : completed
Current execution data :2017-01-10 00:00:00 , count :60, status : completed
Current execution data :2017-01-13 00:00:00 , count :60, status : completed
Current execution data :2017-01-16 00:00:00 , count :60, status : completed
Current execution data :2017-01-19 00:00:00 , count :60, status : completed
Current execution data :2017-01-22 00:00:00 , count :60, status : completed
Current execution data :2017-01-25 00:00:00 , count :60, status : completed
Current execution data :2017-01-28 00:00:00 , count :60, status : completed
Current execution data :2017-01-31 00:00:00 , count :60, status : completed
Current execution data :2017-02-03 00:00:00 , count :60, status : completed
Current execution data :2017-02-06 00:00:00 , count :60, status : completed
Current exec

In [34]:
print(len(df_real))

7260


In [35]:
# Drop duplicated GUID news
df_real_nodup =  df_real.drop_duplicates(subset = ['GUID'], keep='first')

In [36]:
df_real_nodup['date'] = pd.to_datetime(df_real_nodup['PUBLISHED_ON'], unit='s').dt.date

C:\Users\user\AppData\Local\Temp\ipykernel_28152\224847896.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_real_nodup['date'] = pd.to_datetime(df_real_nodup['PUBLISHED_ON'], unit='s').dt.date


In [37]:
df_real_nodup['date'].value_counts().sort_index()

date
2016-12-06    2
2016-12-07    4
2016-12-08    5
2016-12-09    2
2016-12-10    2
             ..
2017-12-20    7
2017-12-21    5
2017-12-22    5
2017-12-23    1
2017-12-24    1
Name: count, Length: 364, dtype: int64

In [38]:
# naming the csv file
min_max_date = df_real_nodup['date'].agg(['min','max'])
df_real_nodup.to_csv(f"data_{min_max_date['min']}_{min_max_date['max']}.csv")

In [39]:
print(f'data_{min_max_date['min']}_{min_max_date['max']}')

data_2016-12-06_2017-12-24
